In [3]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Online Retail.xlsx"

print("Рабочая папка:", Path.cwd())
print("Файл найден:", DATA_PATH.exists())

Рабочая папка: /Users/chirkovgregory/Documents/Projects/online-retail-analysis/notebooks
Файл найден: True


In [4]:
df = pd.read_excel(DATA_PATH, engine='openpyxl')

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [5]:
print("Количество строк: ", df.shape[0])

print("Количество столбцов: ", df.shape[1])

df.info()

Количество строк:  541909
Количество столбцов:  8
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [6]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean().mul(100).round(2),
})

missing.sort_values("missing_count", ascending=False)

,missing_count,missing_percent
CustomerID,135080,24.93
Description,1454,0.27
InvoiceNo,0,0.00
StockCode,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00


In [ ]:
invoice_no = df["InvoiceNo"].astype("string").str.strip().str.upper()

# Маска: True для строк, где номер документа начинается с C.
is_cancellation = invoice_no.str.startswith("C", na=False)

quality_summary = pd.DataFrame({
    "check": [
        "Пропущен CustomerID",
        "Пропущен Description",
        "Полные дубли сверх первого экземпляра",
        "Quantity меньше нуля",
        "Quantity равен нулю",
        "UnitPrice меньше нуля",
        "UnitPrice равен нулю",
        "Номер документа начинается с C",
    ],
    "row_count": [
        df["CustomerID"].isna().sum(),
        df["Description"].isna().sum(),
        df.duplicated().sum(),
        df["Quantity"].lt(0).sum(),
        df["Quantity"].eq(0).sum(),
        df["UnitPrice"].lt(0).sum(),
        df["UnitPrice"].eq(0).sum(),
        is_cancellation.sum(),
    ],
})

quality_summary["row_percent"] = (
    quality_summary["row_count"]
    .div(len(df))
    .mul(100)
    .round(2)
)

quality_summary

,check,row_count,row_percent
0,Пропущен CustomerID,135080,24.93
1,Пропущен Description,1454,0.27
2,Полные дубли сверх первого экземпляра,5268,0.97
3,Quantity меньше нуля,10624,1.96
4,Quantity равен нулю,0,0.00
5,UnitPrice меньше нуля,2,0.00
6,UnitPrice равен нулю,2515,0.46
7,Номер документа начинается с C,9288,1.71


In [8]:
columns_to_show = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    "CustomerID",
]

df.loc[df["Quantity"].lt(0), columns_to_show].head(10)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
141,C536379,D,Discount,-1,27.50,14527.0
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,15311.0
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,17548.0
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,17548.0
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,17548.0
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,17548.0
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,3.45,17548.0
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65,17548.0
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,1.65,17548.0
939,C536506,22960,JAM MAKING SET WITH JARS,-6,4.25,17897.0


In [9]:
df.loc[df["UnitPrice"].le(0), columns_to_show].head(10)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
622,536414,22139,NaN,56,0.0,NaN
1970,536545,21134,NaN,1,0.0,NaN
1971,536546,22145,NaN,1,0.0,NaN
1972,536547,37509,NaN,1,0.0,NaN
1987,536549,85226A,NaN,1,0.0,NaN
1988,536550,85044,NaN,1,0.0,NaN
2024,536552,20950,NaN,1,0.0,NaN
2025,536553,37461,NaN,3,0.0,NaN
2026,536554,84670,NaN,23,0.0,NaN
2406,536589,21777,NaN,-10,0.0,NaN


In [10]:
pd.crosstab(
    df["Quantity"].lt(0).rename("quantity_is_negative"),
    is_cancellation.rename("invoice_starts_with_C"),
)

invoice_starts_with_C,False,True
quantity_is_negative,,
False,531285,0
True,1336,9288


In [11]:
print("Первая операция:", df["InvoiceDate"].min())
print("Последняя операция:", df["InvoiceDate"].max())

Первая операция: 2010-12-01 08:26:00
Последняя операция: 2011-12-09 12:50:00


In [12]:
prepared = df.copy()

prepared["is_duplicate"] = df.duplicated()

print("Строк в исходной таблице:", len(df))
print("Строк в рабочей таблице:", len(prepared))

Строк в исходной таблице: 541909
Строк в рабочей таблице: 541909


In [13]:
prepared["InvoiceNo"] = prepared["InvoiceNo"].astype("string").str.strip().str.upper()

prepared["StockCode"] = prepared["StockCode"].astype("string").str.strip().str.upper()

In [14]:
known_customer_ids = prepared["CustomerID"].dropna()

assert known_customer_ids.mod(1).eq(0).all(), (
    "Найдены CustomerID с дробной частью"
)

prepared["CustomerID"] = (
    prepared["CustomerID"]
    .astype("Int64")
    .astype("string")
)

prepared[["InvoiceNo", "StockCode", "CustomerID"]].head()

,InvoiceNo,StockCode,CustomerID
0,536365,85123A,17850
1,536365,71053,17850
2,536365,84406B,17850
3,536365,84029G,17850
4,536365,84029E,17850


In [15]:
prepared["is_cancellation"] = prepared["InvoiceNo"].str.startswith("C", na=False)

prepared["has_customer"] = prepared["CustomerID"].notna()

prepared["is_sale_candidate"] = (
    ~prepared["is_cancellation"]
    & prepared["Quantity"].gt(0)
    & prepared["UnitPrice"].gt(0)
)

In [16]:
sales_candidates = prepared.loc[
    prepared["is_sale_candidate"]
].copy()

customer_sales_candidates = prepared.loc[
    prepared["is_sale_candidate"]
    & prepared["has_customer"]
].copy()

In [18]:
selection_summary = pd.DataFrame({
    "selection": ["Все строки:",
                  "Кандидаты для анализа продаж:",
                  "Кандидаты с известным покупателем:"],
    "row_count": [len(prepared),
                  len(sales_candidates),
                  len(customer_sales_candidates)],
})

selection_summary["percent_of_source"] = (
    selection_summary["row_count"]
    .div(len(df))
    .mul(100)
    .round(2)
)

selection_summary

,selection,row_count,percent_of_source
0,Все строки:,541909,100.00
1,Кандидаты для анализа продаж:,530104,97.82
2,Кандидаты с известным покупателем:,397884,73.42
